# 03 · LIME vs. atención humana estimada (DeepGaze IIE)

**Pregunta:** ¿las regiones que el clasificador usa para decidir la valencia (LIME) coinciden con las que atraen la mirada humana (DeepGaze IIE)?

Este análisis mide **plausibilidad o alineamiento humano**, no **fidelidad**. Que LIME refleje lo que hace el modelo ya se evalúa en `02_lime.ipynb` (R² local y prueba de aleatorización). DeepGaze es una *predicción* de la atención entrenada con fotografías naturales (MIT1003), no eye-tracking real sobre estos anuncios.

**Mapas de atención** (todos llevados a la geometría que ve LIME y normalizados a suma 1):
| mapa | qué es |
|---|---|
| `deepgaze` | DeepGaze IIE completo (contenido + sesgo central) |
| `deepgaze_content` | DeepGaze con sesgo central uniforme: solo el contenido de la imagen |
| `centerbias` | solo el sesgo central de MIT1003. **Control:** ¿basta con "mirar al centro"? |
| `permutation*` | la explicación de cada imagen contra el mapa de **otras** imágenes. **Control:** ¿es una coincidencia propia de la imagen o una regularidad de composición? |

**Métricas por imagen** (`pl.saliency_alignment`, top-k = superpíxeles con mayor peso LIME positivo):
- `spearman_signed`, `spearman_abs`: pesos LIME (con signo / en valor absoluto) vs. saliencia media de cada superpíxel.
- `mass_topk`, `area_topk`, `enrichment_topk = mass/area`: >1 indica que las regiones que usa el modelo atraen más atención que una región promedio del mismo tamaño.
- `jaccard_topk`: top-k de LIME vs. top-k superpíxeles por saliencia.

In [ ]:
import os, sys

REPO = 'publicity_analysis'
if not os.path.exists(f'/content/{REPO}'):
    !git clone https://github.com/mlacarrasco/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull  # trae la última versión de pipeline.py

os.chdir(f'/content/{REPO}')
sys.path.insert(0, f'/content/{REPO}')

In [ ]:
!pip install -q lime git+https://github.com/matthias-k/DeepGaze.git

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import pipeline as pl

pd.set_option('display.float_format', '{:.3f}'.format)

## 1. Configuración
Los parámetros de LIME (Quick Shift y `num_samples`) se toman de `02_lime.ipynb` si ya se ejecutó. Si no, se usan los valores por defecto de `pl.explain`.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/publicity_analysis/resultados'
except ImportError:
    OUTPUT_DIR = 'resultados'

cfg = pl.Config(originals_dir='images_or', labels_csv='etiquetas_originales.csv',
                output_dir=OUTPUT_DIR)
EXP, SEED = 8, cfg.seeds[0]   # modelo a explicar (el mismo que en 02_lime.ipynb)
ARCH = pl.ARCH_OF_EXP[EXP]
K = 5                         # top-k superpíxeles LIME
N_PERM = 50                   # mapas de otras imágenes por imagen en el baseline de permutación
OUT = os.path.join(cfg.output_dir, 'lime_deepgaze')
os.makedirs(OUT, exist_ok=True)

QS, NUM_SAMPLES = {'kernel_size': 6, 'max_dist': 300, 'ratio': 0.3}, 1000
lime_02 = os.path.join(cfg.output_dir, 'lime')
if os.path.exists(os.path.join(lime_02, 'seleccion_quickshift.csv')):
    best = pd.read_csv(os.path.join(lime_02, 'seleccion_quickshift.csv')).iloc[0]
    QS = {'kernel_size': int(best.kernel_size), 'max_dist': int(best.max_dist), 'ratio': float(best.ratio)}
if os.path.exists(os.path.join(lime_02, 'estabilidad_num_samples.csv')):
    ns = pd.read_csv(os.path.join(lime_02, 'estabilidad_num_samples.csv'))
    stable = ns[ns.jaccard_top5 >= 0.8]
    NUM_SAMPLES = int(stable.num_samples.min() if len(stable) else ns.num_samples.max())
print('Quick Shift:', QS, '| num_samples:', NUM_SAMPLES, '| resultados en', OUT)

## 2. Modelo y conjunto de test

In [ ]:
model = pl.load_trained_model(EXP, SEED, cfg)
predict_fn = pl.make_predict_fn(model, cfg.device)
df = pl.load_labels(cfg)
test_df = pl.make_splits(df, cfg, SEED)['test']
print(len(test_df), 'imágenes de test')

## 3. Explicaciones LIME (todo el test)
Cada explicación se guarda en Drive al terminar. Si la sesión se corta, se retoma desde donde quedó.

In [ ]:
lime_cache = os.path.join(OUT, 'lime', f"exp{EXP}_seed{SEED}_ks{QS['kernel_size']}_md{QS['max_dist']}"
                                       f"_r{QS['ratio']}_n{NUM_SAMPLES}")
explanations = {}
for i, r in enumerate(test_df.itertuples()):
    p = os.path.join(lime_cache, os.path.splitext(r.image)[0] + '.npz')
    if os.path.exists(p):
        explanations[r.image] = pl.load_explanation(p)
        continue
    res = pl.explain(pl.load_image_01(r.path, ARCH, cfg), predict_fn, **QS,
                     num_samples=NUM_SAMPLES, random_state=SEED)
    pl.save_explanation(res, p)
    explanations[r.image] = res
    if (i + 1) % 25 == 0:
        print(f'{i + 1}/{len(test_df)}')
print(len(explanations), 'explicaciones')

## 4. Mapas de atención DeepGaze IIE
DeepGaze se aplica a la imagen **original** (con su relación de aspecto) y el mapa se lleva después a la geometría cuadrada que ve el clasificador.

In [ ]:
dg_model, cb_template = pl.load_deepgaze(cfg, os.path.join(OUT, 'centerbias_mit1003.npy'))
maps = {r.image: pl.deepgaze_maps(r.path, dg_model, cb_template, ARCH, cfg)
        for r in test_df.itertuples()}
del dg_model
if cfg.device == 'cuda':
    pl.torch.cuda.empty_cache()

## 5. Métricas de alineamiento por imagen y baselines

In [ ]:
METRICS = ['spearman_signed', 'spearman_abs', 'enrichment_topk', 'mass_topk', 'jaccard_topk']
label_of = test_df.set_index('image')['label']

rows = []
for name, res in explanations.items():
    base = {'image': name, 'true': pl.CLASS_NAMES[label_of[name]],
            'pred': pl.CLASS_NAMES[res['label']], 'correct': label_of[name] == res['label'],
            'confidence': res['prob'], 'lime_r2': res['score']}
    for mk in ('deepgaze', 'deepgaze_content', 'centerbias'):
        rows.append({**base, 'map': mk, **pl.saliency_alignment(res, maps[name][mk], K)})
align = pd.DataFrame(rows)

info = align.loc[align['map'] == 'deepgaze', ['image', 'true', 'pred', 'correct', 'confidence', 'lime_r2']]
for mk, label in (('deepgaze', 'permutation'), ('deepgaze_content', 'permutation_content')):
    perm = pl.permutation_alignment(explanations, {n: m[mk] for n, m in maps.items()}, K, N_PERM, SEED)
    align = pd.concat([align, info.merge(perm, on='image').assign(map=label)], ignore_index=True)

align.to_csv(os.path.join(OUT, 'alineamiento_por_imagen.csv'), index=False)
align.groupby('map', sort=False)[METRICS].mean()

## 6. ¿El alineamiento supera a los controles?
Wilcoxon pareado por imagen (mapa real vs. control), con corrección de Holm sobre todas las comparaciones de la tabla. `frac_gt` es la fracción de imágenes en que el mapa real supera al control: 0.5 equivale a no haber diferencia.

Se repite solo con las explicaciones más confiables (R² de LIME ≥ mediana): con un R² bajo, la explicación se parece a ruido.

In [ ]:
COMPARISONS = [('deepgaze', 'permutation'), ('deepgaze', 'centerbias'),
               ('deepgaze_content', 'permutation_content'), ('deepgaze_content', 'centerbias')]


def holm(p):
    p = np.asarray(p, dtype=float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    order = np.argsort(p[ok])
    m = ok.sum()
    adj = np.minimum(np.maximum.accumulate((m - np.arange(m)) * p[ok][order]), 1)
    tmp = np.empty(m)
    tmp[order] = adj
    out[ok] = tmp
    return out


def compare(sub):
    rows = []
    for metric in METRICS:
        wide = sub.pivot(index='image', columns='map', values=metric)
        for a, b in COMPARISONS:
            d = (wide[a] - wide[b]).dropna()
            p = stats.wilcoxon(d).pvalue if len(d) >= 10 and (d != 0).any() else np.nan
            rows.append({'metric': metric, 'map': a, 'control': b, 'n': len(d),
                         'mean_map': wide[a].mean(), 'mean_control': wide[b].mean(),
                         'median_diff': d.median(), 'frac_gt': (d > 0).mean(), 'p': p})
    t = pd.DataFrame(rows)
    t['p_holm'] = holm(t['p'])
    return t


tests_all = compare(align)
r2_med = info['lime_r2'].median()
tests_r2 = compare(align[align['lime_r2'] >= r2_med])
tests_all.to_csv(os.path.join(OUT, 'pruebas_todas.csv'), index=False)
tests_r2.to_csv(os.path.join(OUT, 'pruebas_r2_alto.csv'), index=False)
print(f'Todas las imágenes (n={len(info)})')
display(tests_all)
print(f'Solo R² de LIME ≥ {r2_med:.3f}')
display(tests_r2)

## 7. ¿Cambia según la clase predicha, el acierto y la fragmentación?
Conecta con el análisis de fragmentación de `02_lime.ipynb`: ¿las explicaciones fragmentadas aparecen en imágenes donde la atención humana también está dispersa?

In [ ]:
dg = align[align['map'] == 'deepgaze'].copy()
display(dg.groupby('pred')[METRICS].agg(['mean', 'count']))
display(dg.groupby('correct')[METRICS].mean())

for metric in ('enrichment_topk', 'spearman_abs'):
    groups = [g[metric].dropna() for _, g in dg.groupby('pred') if g[metric].notna().sum() >= 2]
    if len(groups) >= 2:
        print(f'Kruskal-Wallis {metric} ~ clase predicha:', stats.kruskal(*groups))


def entropy_norm(m):
    p = m[m > 0].ravel()
    return float(-(p * np.log(p)).sum() / np.log(m.size))


dg['lime_entropy'] = dg['image'].map(lambda n: pl.fragmentation(explanations[n]['weights'])['entropy_norm'])
dg['deepgaze_entropy'] = dg['image'].map(lambda n: entropy_norm(maps[n]['deepgaze_content']))
print('Spearman fragmentación LIME vs dispersión de la atención:',
      stats.spearmanr(dg['lime_entropy'], dg['deepgaze_entropy'], nan_policy='omit'))

## 8. Figuras de ejemplo
Por cada clase predicha, la imagen con mayor R² de LIME. El contorno marca los top-k superpíxeles positivos de LIME.

In [ ]:
from skimage.segmentation import find_boundaries

examples = (dg.sort_values('lime_r2', ascending=False).groupby('pred').head(1)['image'].tolist())
fig, axes = plt.subplots(len(examples), 4, figsize=(15, 3.8 * len(examples)), squeeze=False)
path_of = test_df.set_index('image')['path']
for row, name in zip(axes, examples):
    res = explanations[name]
    img = pl.load_image_01(path_of[name], ARCH, cfg)
    top = find_boundaries(np.isin(res['segments'], pl.topk_segments(res['weights'], K)))
    wm = pl.weight_map(res)
    lim = np.abs(wm).max()
    r = dg.set_index('image').loc[name]
    row[0].imshow(img)
    row[0].set_title(f"True: {r['true']} | Pred: {r['pred']} ({r['confidence']:.2f})")
    im = row[1].imshow(wm, cmap='RdBu', vmin=-lim, vmax=lim)
    row[1].set_title(f"LIME (R²={r['lime_r2']:.2f})")
    plt.colorbar(im, ax=row[1], fraction=0.046, label='LIME weight')
    for ax, mk, title in ((row[2], 'deepgaze', 'DeepGaze IIE'),
                          (row[3], 'deepgaze_content', 'DeepGaze (sin sesgo central)')):
        ax.imshow(img)
        ax.imshow(maps[name][mk], cmap='magma', alpha=0.6)
        ax.set_title(f"{title}\nenrichment={align[(align.image == name) & (align.map == mk)]['enrichment_topk'].iloc[0]:.2f}")
    for ax in row:
        ax.contour(top, levels=[0.5], colors='cyan', linewidths=1)
        ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'ejemplos_lime_deepgaze.png'), dpi=200)
plt.show()

## 9. Cómo leer los resultados
- **`deepgaze` > `permutation` y > `centerbias`**: el modelo se apoya en las regiones que atraen la mirada *en esa imagen concreta*, más allá de la composición típica de los anuncios. Es un argumento de plausibilidad.
- **`deepgaze` ≈ `centerbias`**: el alineamiento se explica solo porque ambos favorecen el centro. Hay que reportar `deepgaze_content`.
- **`deepgaze` ≈ `permutation`**: lo que coincide es la composición habitual de los anuncios, no el contenido de cada imagen.
- **Alineamiento bajo**: el modelo usa regiones que la gente no mira (fondo, bordes, texturas), o la valencia depende de elementos que no atraen la mirada. Es un hallazgo, no un error.
- **Limitaciones a declarar**: DeepGaze es una predicción entrenada con fotografías naturales (MIT1003), no eye-tracking real sobre estos anuncios; en anuncios con mucho texto puede ser menos preciso.